In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# 1. Clean up resources: Stop any existing active Spark session in memory to prevent session reuse conflicts
if "spark" in locals() and spark:
    spark.stop()

# 2. Start building the SparkSession configuration using the builder pattern
builder = (
    SparkSession.builder.appName("DeltaLakeMastery")
    
    # Set execution mode to local, using all available CPU cores/threads (*)
    .master("local[*]")
    
    # Register Delta Lake SQL extensions to enable ACID commands (MERGE, UPDATE, DELETE, OPTIMIZE, VACUUM)
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    
    # Override the default Spark catalog to route table operations through DeltaCatalog
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    
    # Pull the Delta Lake package dependencies from Maven Central (targeting Scala 2.12 / Delta 3.2.0)
    # .config("spark.jars.packages", "io.delta:delta-spark_2.12:4.0.0")

    # Define the local warehouse directory where Spark stores managed table metadata and data files
    .config("spark.sql.warehouse.dir", "/workspace/data/warehouse")
)

# 3. Inject Delta-pip configurations into the builder and instantiate the final SparkSession
# Note: configure_spark_with_delta_pip automatically handles registering the correct Delta JARs 
# downloaded via pip alongside Maven, ensuring smooth integration.
spark = configure_spark_with_delta_pip(builder).getOrCreate()

# 4. Reduce log verbosity: Silence INFO-level messages, keeping only WARN and ERROR logs clean in your console
spark.sparkContext.setLogLevel("WARN")

# 5. Verify and print the running runtime versions and application identifiers
print("Spark Version:", spark.version)
print("Application:", spark.sparkContext.appName)

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/notebook/.ivy2.5.2/cache
The jars for the packages stored in: /home/notebook/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4dd0cd98-b98c-4f8d-9cd7-a275f9513e9a;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.

Spark Version: 4.2.0
Application: DeltaLakeMastery


This script is the foundational blueprint for running **Delta Lake** locally in your Docker container. It configures a Apache Spark session with all the necessary extensions, catalogs, and packages required to perform ACID transactions (like `MERGE`, `UPDATE`, `DELETE`, `OPTIMIZE`, and `VACUUM`).

---

### **Line-by-Line Breakdown**

#### **1. Imports and Session Cleanup**

```python
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

if "spark" in locals() and spark:
    spark.stop()

```

* **Imports**: Brings in the core `SparkSession` class and Delta's official Python helper (`configure_spark_with_delta_pip`).
* **Cleanup (`spark.stop()`)**: Checks if a previous Spark session is already running in memory and shuts it down. This prevents configuration conflicts or JVM reuse errors if you re-run the cell.

---

#### **2. The SparkSession Builder Pattern**

```python
builder = (
    SparkSession.builder.appName("DeltaLakeMastery")
    .master("local[*]")
...

```

* **`.appName("DeltaLakeMastery")`**: Names your application. This identifier appears in the Spark Web UI so you can track job execution and performance metrics.
* **`.master("local[*]")`**: Instructs Spark to run locally on your machine using **all available CPU cores/threads** (`[*]`), maximizing the compute power of your local Docker container.

---

#### **3. Registering Delta Extensions and Catalog**

```python
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.sys... DeltaCatalog",
    )

```

* **`spark.sql.extensions`**: Injects Delta's custom SQL parser and command processor into Spark. Without this, Spark SQL would throw syntax errors when encountering Delta-specific commands like `OPTIMIZE`, `VACUUM`, or advanced `MERGE INTO` syntax.
* **`spark.sql.catalog.spark_catalog`**: Overrides Spark's default table catalog with Delta's `DeltaCatalog`. This ensures that any standard table you create via SQL or PySpark defaults to the **Delta format** instead of legacy Parquet or Hive formats.

---

#### **4. Managing Dependencies and Storage**

```python
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:4.0.0")
    .config("spark.sql.warehouse.dir", "/workspace/data/warehouse")
)

```

* **`spark.jars.packages`**: Automatically downloads and hooks up the required Delta Lake Java/Scala binaries from Maven Central matching your runtime version.
* **`spark.sql.warehouse.dir`**: Defines the default directory where Spark stores managed table metadata and data files on your Docker container's file system (`/workspace/data/warehouse`).

---

#### **5. Instantiating and Tuning the Session**

```python
spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("Spark Version:", spark.version)
print("Application:", spark.sparkContext.appName)

```

* **`configure_spark_with_delta_pip(builder)`**: A crucial utility function when working with pip-installed Delta packages. It synchronizes the Python package binaries with Spark's underlying JVM classpath to avoid runtime linkage errors.
* **`setLogLevel("WARN")`**: Suppresses verbose `INFO`-level logs in your console, keeping your output clean and focusing only on warnings and errors.
* **Verification**: Prints out the active runtime version of Spark and the application name to confirm everything initialized successfully.

# **Create**

### **Showing Avaliable DataBase**

In [8]:
#PySpark
database_df = spark.catalog.listDatabases()


for db in database_df:
    print(f"Database : {db.name} | Description : {db.description} | Path : {db.locationUri}")

Database : default | Description : default database | Path : file:/workspace/data/warehouse


In [9]:
#PySpark
spark.sql("SHOW DATABASES").show(truncate=False)

+---------+
|namespace|
+---------+
|default  |
+---------+



In [10]:
#PySpark
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|  default|
+---------+



In [15]:
#SQL

spark.sql("SELECT current_database()").show()

+----------------+
|current_schema()|
+----------------+
|         default|
+----------------+



### **Create a DataBase**

In [17]:
#PySpark
# Create database (without the existsOrDefault argument)
spark.catalog.createDatabase("exampleDB_python", ifNotExists=True)

# Set current database
spark.catalog.setCurrentDatabase("exampleDB_python")

# Verify current database
print(f"Current Working Database: {spark.catalog.currentDatabase()}")

# List all databases
for db in spark.catalog.listDatabases():
    print(db.name)

Current Working Database: exampleDB_python
default
exampledb_python


26/11/04 07:52:28 WARN CreateNamespaceExec: Namespace exampleDB_python was created concurrently. Ignoring.


In [19]:
#SQL

spark.sql("CREATE DATABASE IF NOT EXISTS exampleDB_sql")

spark.sql("USE exampleDB_sql")

spark.sql("SELECT current_database()").show()

+----------------+
|current_schema()|
+----------------+
|   exampleDB_sql|
+----------------+



## **Creating a Delta Lake Table**

In [22]:
#SQL

spark.sql("""
    CREATE TABLE IF NOT EXISTS exampleDB_sql.countries (
        id LONG,
        country STRING,
        capital STRING
    ) USING DELTA
""")

DataFrame[]

In [25]:
#python
from pyspark.sql.types import *
from delta.tables import *

delta_table = (
    DeltaTable.create(spark)
    .tableName("exampleDB_python.countries")
    .addColumn("id", dataType=LongType(), nullable=False)
    .addColumn("country", dataType=StringType(), nullable=False)
    .addColumn("capital", dataType=StringType(), nullable=False)
    .execute()
)

### **Showing avaliable Table**

In [33]:
#sql
spark.sql("SHOW TABLES IN exampleDB_sql").show()

spark.sql("DESCRIBE EXTENDED exampleDB_sql.countries").show(truncate=False)

spark.sql("DESCRIBE HISTORY exampleDB_sql.countries").show(truncate=False)

+-------------+---------+-----------+
|    namespace|tableName|isTemporary|
+-------------+---------+-----------+
|exampledb_sql|countries|      false|
+-------------+---------+-----------+

+----------------------------+---------------------------------------------------------+-------+
|col_name                    |data_type                                                |comment|
+----------------------------+---------------------------------------------------------+-------+
|id                          |bigint                                                   |NULL   |
|country                     |string                                                   |NULL   |
|capital                     |string                                                   |NULL   |
|                            |                                                         |       |
|# Detailed Table Information|                                                         |       |
|Catalog                     |spa

In [39]:
#pyspark
table_exists = spark.catalog.tableExists("exampleDb_python.countries")
print(f"Table exists: {table_exists}")

print(spark.catalog.listTables("exampleDB_python"))

spark.table("exampleDB_sql.countries").show()

Table exists: True
[Table(name='countries', catalog='spark_catalog', namespace=['exampledb_python'], description=None, tableType='MANAGED', isTemporary=False)]
+---+-------+-------+
| id|country|capital|
+---+-------+-------+
+---+-------+-------+



## **Loading Data into a Delta Lake Table**